# DHS aggregate immunization research

The DHS Program Indicator Data API, The Demographic and Health Surveys (DHS) Program. ICF. Originally funded by the United States Agency for International Development (USAID). Available from https://api.dhsprogram.com/. See fetch_dates for access times. SurveyBridge AI is independent of The DHS Program.

Read limitations.md. Survey-specific comparability is unverified. No interpolation or inference.

In [ ]:
from pathlib import Path
import json, hashlib
import pandas as pd
root = Path.cwd()
manifest = json.loads((root / 'provenance.json').read_text(encoding='utf-8'))
for name, expected in manifest['output_checksums'].items():
    assert hashlib.sha256((root / name).read_bytes()).hexdigest() == expected, name
project = json.loads((root / 'project.json').read_text(encoding='utf-8'))
from surveybridge.services.provenance import verify_project
assert verify_project(project)
print('All archived response and output hashes verified')

In [ ]:
from surveybridge.services.series import normalize
from surveybridge.models import Indicator
source_records = []
for page in project['pages']:
    if '/data?' not in page['url']:
        continue
    for raw in json.loads(bytes.fromhex(page['body_hex']))['Data']:
        from surveybridge.utils.cache import canonical, digest
        source_records.append(dict(raw=raw, source_url=page['url'], retrieved_utc=page['retrieved_utc'], source_response_hash=page['sha256'], source_record_hash=digest(canonical(raw))))
rebuilt, _ = normalize(source_records, {i['id']: Indicator(**i) for i in project['indicators']})
expected = sorted(project['observations'], key=lambda r: r['source_row_id'])
assert sorted(rebuilt, key=lambda r: r['source_row_id']) == expected
df = pd.DataFrame(json.loads((root / 'observations.json').read_text(encoding='utf-8')))
print('Every normalized observation reproduced from original API records')
df[['source_country_name', 'source_survey_id', 'survey_year_label', 'indicator_id', 'value', 'unit']]

In [ ]:
report = json.loads((root / 'comparability_report.json').read_text(encoding='utf-8'))
for finding in report['findings']:
    print(finding['severity'].upper(), finding['explanation'])
by_indicator = json.loads((root / 'comparability_by_indicator.json').read_text(encoding='utf-8'))
for indicator_id, check in by_indicator.items():
    print(indicator_id, 'Descriptive source table; publication comparability unverified')
    if not check['comparison_allowed']:
        print('BLOCKED: review indicator-specific findings before comparison')
        continue
    subset = df[df.indicator_id == indicator_id]
    display(subset[['source_country_name', 'source_survey_id', 'survey_year_label', 'value', 'unit']])
# Keep indicators separate. No annual interpolation, pooling, confidence intervals or significance tests.